# Case Study: Conformance Testing of a TA Implementation Against a TRE Specification

This notebook accompanies the *language inclusion* experiment from the paper.
We demonstrate the sample-and-match approach in the cross-formalism setting:
given a TA implementation and a TRE specification, we check L(TA) ⊆ L(φ).

**Setup** — arbiter property over alphabet {r, g}, slice n=10, T=10:
- **φ₁** — every request `r` is eventually followed by a grant `g`

We study two candidate TA implementations of φ₁:
- **TA₁** (`ta_1.prism`) — bug: `label "accepting" = true` (both states accepting)
- **TA₂** (`ta_2.prism`) — fix: `label "accepting" = state=0`

**Procedure**: sample words from the TA via wordgen, check each against φ₁.
Stop as soon as one of two conditions is met:
- a counterexample is found → inclusion **refuted**
- the Clopper-Pearson bound $1 - (\alpha/2)^{1/N}$ drops below a target → inclusion **supported**

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))

In [2]:
from parse.quickparse import quickparse
from match.match import match
from sample.TimedWord import TimedWord
from scipy.stats import beta as beta_dist
import numpy as np

## 1. TRE Specification

In [3]:
spec_dir = '..'

phi1 = quickparse(os.path.join(spec_dir, 'spec_02_subset_A.tre'))
phi2 = quickparse(os.path.join(spec_dir, 'spec_02_subset_B.tre'))

print('phi_1:', phi1.getText())
print('phi_2:', phi2.getText())

phi_1: <(a+g)*.((r.(a+r+g)*.g).(a+g)*)*>_[0,100]
phi_2: <(a+g)*.(<r.(a)*.g>_[1,2].(a+g)*)*>_[0,100]


## 2. TA₁ — Flawed Implementation

`ta_1.prism` declares `label "accepting" = true`, meaning **both** states are accepting.
Words that end in `state=1` — with an unanswered trailing `r` — are accepted by TA₁
even though they violate φ₁.

We check each sampled word against φ₁ and stop at the first counterexample.

In [4]:
words_ta1 = TimedWord.from_wordgen_file('ta1_sample.txt')

# stop at first counterexample
first_cex = None
for w in words_ta1:
    if match(w, phi1) == 0:
        first_cex = w
        break

if first_cex:
    print(f'Counterexample found (last symbol: {first_cex.symbols[-1]}):')
    print(f'  {first_cex}')
    print()
    print('→ L(TA₁) ⊄ L(φ₁)')

Counterexample found (last symbol: r):
  (1.529, r),(0.415, g),(1.266, r),(0.925, g),(0.313, g),(0.338, g),(0.319, g),(1.704, g),(0.151, g),(3.041, r)

→ L(TA₁) ⊄ L(φ₁)


The first sampled word already ends with a trailing `r` — the root cause of the bug.

## 3. TA₂ — Fixed Implementation

With `label "accepting" = state=0`, only runs returning to the idle state are accepted.
We sample from TA₂ directly (no filtering) and accumulate the Clopper-Pearson bound,
stopping once it drops below a chosen threshold.

In [5]:
alpha_half = 0.05  # 95% confidence

def cp_bound(N):
    return 1 - alpha_half ** (1 / N)

words_ta2 = TimedWord.from_wordgen_file('ta2_sample.txt')

target = 0.03
n_target = None
for i, w in enumerate(words_ta2):
    if match(w, phi1) == 0:
        print(f'Counterexample at sample #{i+1} — inclusion REFUTED')
        break
    N = i + 1
    if n_target is None and cp_bound(N) < target:
        n_target = N
        print(f'Bound dropped below {target} at N={N}  (p_fail < {cp_bound(N):.4f}, 95% confidence)')
        print(f'→ L(TA₂) ⊆ L(φ₁)  [statistically supported]')
        break
else:
    if n_target is None:
        N = len(words_ta2)
        print(f'No counterexample in {N} samples  |  p_fail < {cp_bound(N):.4f}  at 95% confidence')

print()
print('How the bound tightens (all samples passing):')
for N in [29, 59, 99]:
    print(f'  N={N:>3}  →  p_fail < {cp_bound(N):.4f}')

Bound dropped below 0.03 at N=99  (p_fail < 0.0298, 95% confidence)
→ L(TA₂) ⊆ L(φ₁)  [statistically supported]

How the bound tightens (all samples passing):
  N= 29  →  p_fail < 0.0981
  N= 59  →  p_fail < 0.0495
  N= 99  →  p_fail < 0.0298
